In [1]:
import pandas as pd
import json

data_folder_path = "/Users/lukajin/PycharmProjects/soamp/data/"

In [2]:
df = pd.read_csv(data_folder_path + "mic_classification_dataset.csv")

with open(data_folder_path + "val_split.json", 'r') as f:
    data = json.load(f)

val_ids = data['val_peptide_ids']

df.loc[df['peptide_id'].astype(str).isin(val_ids), 'split'] = 'val'

In [3]:
df.head()

,peptide_id,sequence,smiles,organism,ncbi_taxon_id_if_available,mic_value_uM,mic_type,has_noncanonical,label,split
0,10,LFIFFF,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,Staphylococcus aureus,1280,15.60,censored,False,active,val
1,11,RVKRVWPLVIRTVIAGYNLYRAIKKK,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](CCCN...,Escherichia coli,562,2.24,averaged,False,active,train
2,11,RVKRVWPLVIRTVIAGYNLYRAIKKK,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](CCCN...,Pseudomonas aeruginosa,287,5.06,averaged,False,active,train
3,11,RVKRVWPLVIRTVIAGYNLYRAIKKK,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](CCCN...,Staphylococcus aureus,1280,1.11,averaged,False,active,train
4,12,RKRIHIGPGRAFYTT,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1cnc[nH]1)NC(=O)...,Escherichia coli,562,6.76,averaged,False,active,test


In [4]:
df['split'].value_counts()

split
train    11308
test      7587
val       5735
Name: count, dtype: int64

In [12]:
from qmap.toolkit.clustering import build_graph

from typing import Optional, List, Any, Union
import numpy as np

sequences = df.loc[df['split'] != "test", 'sequence'].unique()

threshold: float = 0.60
random_state: Optional[int] = 42
n_iterations: int = -1

matrix: str = "blosum45"
gap_open: int = 5
gap_extension: int = 1
use_cache: bool = True
verbose: bool = True
num_threads: Optional[int] = None

# Step 2: Build the graph
g, edgelist = build_graph(sequences,
                threshold=threshold,
                matrix=matrix,
                gap_open=gap_open,
                gap_extension=gap_extension,
                use_cache=use_cache,
                show_progress=verbose,
                num_threads=num_threads
                )

print("Number of node:", g.vcount()) if verbose else None
print("Number of edges:", g.ecount()) if verbose else None
print(f"Connection ratio: {100 * g.ecount() / (g.vcount() * (g.vcount() - 1) / 2):.4f}%") if verbose else None

Number of node: 7891
Number of edges: 96693
Connection ratio: 0.3106%


Loaded edgelist from cache: /Users/lukajin/Library/Caches/pwiden_engine/edgelist_de71b7ae2a8230afe51ebb60af75bb44e1be696b049d9339fcbb4d042fda32f5_thresh_0.6000.bin


In [13]:
from qmap.toolkit.clustering import leiden_community_detection

# Step 3: Retrieve the clusters
clusters = leiden_community_detection(g, n_iterations=n_iterations, seed=random_state)

In [17]:
clusters.sort_values(by='node_id', inplace=True)
clusters.reset_index(drop=True, inplace=True)

In [19]:
clusters["sequence"] = sequences

In [22]:
clusters

,node_id,community,sequence
0,0,26,LFIFFF
1,1,51,RVKRVWPLVIRTVIAGYNLYRAIKKK
2,2,415,DSHAKRHHGYKRKFHEKHHSHRGY
3,3,416,ENREVPPGFTALIKTLRKCKII
4,4,89,GMASKAGAIAGKIAKVALKAL
...,...,...,...
7886,7886,5,GFKRIVQRIKDKLRNLV
7887,7887,5,GFKRIVQRIKDFKRNLV
7888,7888,5,GFKRIVQRIKDFLRKLV
7889,7889,5,GFKRIVQRIKDFLRNKV


In [7]:
vs = pd.DataFrame(clusters['community'].value_counts())
vs = vs.sample(frac=1, random_state=random_state)
vs['cumsum'] = vs['count'].cumsum()
vs['fold'] = pd.cut(vs['cumsum'], bins=5, labels=False).astype(str)

In [23]:
fold_ids = {}

for fold in vs['fold'].unique():
    fold_ids[fold] = list(vs.loc[vs['fold'] == fold].index)

def fold_gen(x, dic=fold_ids):
    for k, v in dic.items():
        if x in v:
            return k

In [24]:
clusters['fold_id'] = clusters['community'].apply(fold_gen)

In [26]:
clusters['fold_id'].value_counts()

fold_id
1    1628
3    1608
4    1590
2    1536
0    1529
Name: count, dtype: int64

In [27]:
clusters.to_csv(data_folder_path + "train_folds_leiden.csv", index=False)